# stackoverflow-spark-analysis Project

**Dataset:**
https://www.kaggle.com/datasets/stackoverflow/stack-overflow-2018-developer-survey

**Objectives:**

1. Process: the large-scale dataset to handle multi-value, missing data, and categorical encoding.

2. Analyze: correlations between diverse academic backgrounds and the adoption of special technical roles.

3. Map: the transition patterns between current used technologies and tools developers aim to learn next.

4. Develop: a classification model to predict a developer's role based on their technical profile.

## Phase 3 - RDD Operations

## Setup Spark Environment

In [6]:
import os
import sys

spark_paths = [p for p in sys.path if 'spark' in p.lower()]
for p in spark_paths:
    if p in sys.path:
        sys.path.remove(p)

!pip install pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('StackOverflow RDD Analysis') \
    .master('local[*]') \
    .getOrCreate()

print('Spark version:', spark.version)


Spark version: 4.0.2


## Load Preprocessed Data

In [7]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/Big_Data_Project/stackoverflow_final_transformed.csv" ./

# Read the preprocessed file (output of Phase 2)
df = spark.read.csv('stackoverflow_final_transformed.csv', header=True, inferSchema=True)
df.show(5)

print(f'Rows: {df.count():,} | Cols: {len(df.columns)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+--------------+--------------------+------------------+----------------+-----------------+------+--------------------+--------------------+--------------------+--------------------+-----------------+----------------+---------------+--------------------+-----------------+------------------+-----------------------+---------------+------------------+--------------+----------------------+-----------+-------------------+-----------------------+
|       Country|             DevType|        Employment|     YearsCoding|              Age|Gender|     JobSatisfaction|      UndergradMajor|      EducationTypes|  LanguageWorkedWith|  ConvertedSalary| DevType_segment|Languages_count|EducationTypes_count|       Salary_num|        Salary_log|DevType_segment_indexed|Country_indexed|Employment_indexed|Gender_indexed|UndergradMajor_indexed|Age_encoded|YearsCoding_encoded|JobSat

## Convert DataFrame to RDD

We convert the preprocessed DataFrame into an RDD to enable low-level distributed operations.
Each element in the RDD represents one full survey respondent row.

In [8]:
# Convert DataFrame to RDD
rdd = df.rdd

print('RDD type:', type(rdd))
print('Sample (first 2 rows):')
for row in rdd.take(2):
    print(row)


RDD type: <class 'pyspark.core.rdd.RDD'>
Sample (first 2 rows):
Row(Country='Kenya', DevType='Full-stack developer', Employment='Employed part-time', YearsCoding='3-5 years', Age='25 - 34 years old', Gender='Male', JobSatisfaction='Extremely satisfied', UndergradMajor='Mathematics or statistics', EducationTypes='Taught yourself a new language, framework, or tool without taking a formal course;Participated in a hackathon', LanguageWorkedWith='JavaScript;Python;HTML;CSS', ConvertedSalary=97323.09732339124, DevType_segment='Web', Languages_count=4, EducationTypes_count=2, Salary_num=97323.09732339124, Salary_log=11.485791622566431, DevType_segment_indexed=0.0, Country_indexed=57.0, Employment_indexed=3.0, Gender_indexed=0.0, UndergradMajor_indexed=5.0, Age_encoded=0.0, YearsCoding_encoded=0.0, JobSatisfaction_encoded=1.0)
Row(Country='United Kingdom', DevType='Database administrator;DevOps specialist;Full-stack developer;System administrator', Employment='Employed full-time', YearsCoding=

---
# RDD Transformations

Transformations are **lazy** operations — they define what to do but do not execute until an action is called.

## Transformation 1 - map
**Question:** What is the salary of each developer and which country are they from?

We extract only the `Country` and `Salary_num` fields from each row,
creating a simplified `(country, salary)` pair for each respondent.
This reduces the data to what is needed for salary-based geographic analysis.

In [9]:
# map: Extract (Country, Salary) pairs
countryToSalary = rdd.map(lambda row: (
    row['Country'],
    float(row['Salary_num']) if row['Salary_num'] is not None else 0.0
))

print('Sample output - (Country, Salary):')
for item in countryToSalary.take(5):
    print(item)


Sample output - (Country, Salary):
('Kenya', 97323.09732339124)
('United Kingdom', 70841.0)
('United States', 97323.09732339124)
('South Africa', 21426.0)
('United Kingdom', 41671.0)


## Transformation 2 - filter
**Question:** Which developers earn above the average salary?

We first compute the mean salary, then filter the RDD to keep only
respondents whose salary exceeds this threshold.
This helps identify high-earning developers in the ecosystem.

In [10]:
from pyspark.sql import functions as F

# Calculate average salary from DataFrame
avg_salary = df.agg(F.avg('Salary_num')).first()[0]
print(f'Average salary: ${avg_salary:,.2f}')

# filter: Keep only developers earning above average
high_earners = rdd.filter(lambda row:
    row['Salary_num'] is not None and float(row['Salary_num']) > avg_salary
)

print(f'\nDevelopers above average salary: {high_earners.count():,}')
print('Sample:')
for row in high_earners.take(3):
    print(f"  Country: {row['Country']} | DevType: {row['DevType_segment']} | Salary: ${float(row['Salary_num']):,.2f}")


Average salary: $80,315.76

Developers above average salary: 50,358
Sample:
  Country: Kenya | DevType: Web | Salary: $97,323.10
  Country: United States | DevType: Web | Salary: $97,323.10
  Country: United States | DevType: Web | Salary: $120,000.00


## Transformation 3 - flatMap
**Question:** What individual programming languages do developers use?

The `LanguageWorkedWith` column contains multiple languages per respondent (e.g., `"Python;JavaScript;SQL"`).
We use `flatMap` to split each entry and emit one language per element,
enabling frequency analysis across the full developer population.

In [11]:
# flatMap: Split multi-value language field into individual language entries
languages = rdd.flatMap(lambda row:
    row['LanguageWorkedWith'].split(';') if row['LanguageWorkedWith'] else []
)

print('Sample individual languages:')
for lang in languages.take(10):
    print(' ', lang)


Sample individual languages:
  JavaScript
  Python
  HTML
  CSS
  JavaScript
  Python
  Bash/Shell
  C#
  JavaScript
  SQL


## Transformation 4 - reduceByKey
**Question:** How many developers use each programming language?

Building on Transformation 3, we assign a count of 1 to each language occurrence,
then use `reduceByKey` to sum counts per language.
The result is sorted descending to reveal the most widely-used technologies.

In [12]:
# reduceByKey: Count occurrences of each language
top_languages = (
    languages
    .map(lambda lang: (lang.strip(), 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

print('Top 10 most used programming languages:')
for lang, count in top_languages.take(10):
    print(f'  {lang:<20} {count:,}')


Top 10 most used programming languages:
  JavaScript           56,713
  HTML                 55,473
  CSS                  52,816
  SQL                  46,211
  Java                 36,916
  Bash/Shell           31,253
  Python               30,286
  C#                   27,473
  PHP                  24,684
  C++                  19,706


## Transformation 5 - reduceByKey (Average Salary per DevType)
**Question:** Which developer role earns the highest average salary?

We map each respondent to `(DevType_segment, (salary, 1))`,
then use `reduceByKey` to accumulate total salary and count per segment.
Dividing gives us the mean salary per role — a key insight for understanding
the economic landscape of the developer ecosystem.

In [13]:
# reduceByKey: Compute average salary per DevType segment
salary_by_devtype = (
    rdd
    .map(lambda row: (
        row['DevType_segment'],
        (float(row['Salary_num']) if row['Salary_num'] else 0.0, 1)
    ))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    .map(lambda x: (x[0], x[1][0] / x[1][1]))
    .sortBy(lambda x: x[1], ascending=False)
)

print('Average salary by Developer Type (sorted highest to lowest):')
for devtype, avg_sal in salary_by_devtype.collect():
    print(f'  {devtype:<20} ${avg_sal:,.2f}')


Average salary by Developer Type (sorted highest to lowest):
  Other                $85,621.63
  Data                 $83,977.96
  Web                  $80,675.20
  Mobile               $77,506.71
  Student/Academic     $77,282.99


---
# RDD Actions

Actions **trigger execution** of all pending transformations and return a result.

## Action 1 - count
**What it reveals:** The total number of developer records in the preprocessed dataset.
This confirms the dataset size after all cleaning steps in Phase 2.

In [14]:
# count: Total number of records
total_count = rdd.count()
print(f'Total developer records: {total_count:,}')


Total developer records: 81,582


## Action 2 - first
**What it reveals:** The very first respondent record in the dataset.
Useful for quickly inspecting the structure of the data after preprocessing.

In [15]:
# first: Retrieve the first row
first_row = rdd.first()
print('First record in the dataset:')
print(f"  Country    : {first_row['Country']}")
print(f"  DevType    : {first_row['DevType_segment']}")
print(f"  Employment : {first_row['Employment']}")
print(f"  Salary     : ${float(first_row['Salary_num']):,.2f}")
print(f"  Languages  : {first_row['LanguageWorkedWith']}")


First record in the dataset:
  Country    : Kenya
  DevType    : Web
  Employment : Employed part-time
  Salary     : $97,323.10
  Languages  : JavaScript;Python;HTML;CSS


## Action 3 - reduce
**What it reveals:** The total sum of all developer salaries in the dataset.
This gives a macro-level view of global developer compensation captured in this survey.

In [16]:
# reduce: Sum all salaries
total_salary = (
    rdd
    .map(lambda row: float(row['Salary_num']) if row['Salary_num'] else 0.0)
    .reduce(lambda a, b: a + b)
)

print(f'Total salary across all developers: ${total_salary:,.2f}')
print(f'(Approximately ${total_salary/1e9:.2f} billion USD)')


Total salary across all developers: $6,552,320,656.01
(Approximately $6.55 billion USD)


## Action 4 - take
**What it reveals:** The first 10 developers classified as Data scientists/analysts.
This verifies that the DevType segmentation from Phase 2 correctly identifies Data professionals.

In [17]:
# take: Get first 10 Data segment developers
data_devs = rdd.filter(lambda row: row['DevType_segment'] == 'Data').take(10)

print('First 10 Data developers:')
for i, row in enumerate(data_devs, 1):
    langs = row['LanguageWorkedWith'][:40] if row['LanguageWorkedWith'] else 'N/A'
    print(f"  {i}. Country: {row['Country']:<15} | Languages: {langs}...")


First 10 Data developers:
  1. Country: India           | Languages: C;C++;C#...
  2. Country: Finland         | Languages: JavaScript;Python;R;SQL;VBA;HTML;CSS...
  3. Country: India           | Languages: C;C++;Java;HTML;CSS...
  4. Country: Italy           | Languages: Java;JavaScript;Scala;SQL;HTML;CSS...
  5. Country: United States   | Languages: JavaScript;Python;SQL;VBA...
  6. Country: United States   | Languages: Python;R...
  7. Country: India           | Languages: C;C++;CoffeeScript;Haskell;Java;JavaScri...
  8. Country: United States   | Languages: Groovy;Python;R;SQL;HTML;CSS;Bash/Shell...
  9. Country: France          | Languages: C++;Python;R...
  10. Country: Canada          | Languages: JavaScript;Python;R;SQL;HTML;CSS...


## Action 5 - collect
**What it reveals:** The complete ranked list of average salaries by developer type.
Since there are only 5 segments, `collect` is safe here and gives us the full picture
of compensation across the developer ecosystem.

In [18]:
# collect: Bring all (DevType, AvgSalary) results to driver
result = salary_by_devtype.collect()

print('Complete average salary ranking by Developer Type:')
print('-' * 45)
for rank, (devtype, avg_sal) in enumerate(result, 1):
    print(f'  #{rank}  {devtype:<20} ${avg_sal:,.2f}')
print('-' * 45)


Complete average salary ranking by Developer Type:
---------------------------------------------
  #1  Other                $85,621.63
  #2  Data                 $83,977.96
  #3  Web                  $80,675.20
  #4  Mobile               $77,506.71
  #5  Student/Academic     $77,282.99
---------------------------------------------


---
# Summary of RDD Operations

## Key Insights

**From Transformations:**
- **map** revealed that salary and country can be effectively extracted for geographic compensation analysis.
- **filter** showed that 50,358 out of 81,582 developers (61.7%) earn above the average salary.
- **flatMap** successfully decomposed multi-value language fields into individual entries for frequency analysis.
- **reduceByKey (languages)** confirmed that JavaScript (56,713), HTML (55,473), and CSS (52,816) are the top 3 languages — consistent with the dominant Web segment in our dataset.
- **reduceByKey (salary)** revealed that Other roles ($85,621) and Data scientists ($83,977) earn the most on average.

**From Actions:**
- **count** confirmed 81,582 clean records after Phase 2 preprocessing.
- **first** validated the row structure is correct and consistent.
- **reduce** computed a total salary pool of approximately $6.55 billion USD across all respondents.
- **take** verified the DevType segmentation correctly identifies Data professionals.
- **collect** provided the complete salary ranking across all 5 developer segments.

## Connection to Problem Statement
These RDD operations support our core objective of understanding the Global Developer Ecosystem.
The language distribution findings directly explain why Web developers dominate the dataset (55,316 respondents),
while salary analysis by DevType provides foundational evidence for our ML classification goal:
predicting a developer's role based on their technical and demographic profile.